In [1]:
print("AI Resume Screening System")
print("Welcome to our Hackathon Project!")

import sys
print("Python version:", sys.version)

AI Resume Screening System
Welcome to our Hackathon Project!
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [1]:
!pip -q install pymupdf pandas scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SKILLS = [
    "python", "java", "c++", "sql", "html", "css",
    "javascript", "flask", "django", "machine learning",
    "deep learning", "data analysis", "git", "github",
    "react", "pandas", "numpy", "tensorflow"
]

def extract_skills(text):
    text = text.lower()
    found = []
    for skill in SKILLS:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            found.append(skill)
    return found

def screen_resume(resume_text, job_description):
    resume_skills = set(extract_skills(resume_text))
    job_skills = set(extract_skills(job_description))

    matched = sorted(resume_skills & job_skills)
    missing = sorted(job_skills - resume_skills)

    score = (
        round(100 * len(matched) / len(job_skills))
        if job_skills else 0
    )

    return {
        "Match Score (%)": score,
        "Matched Skills": ", ".join(matched) or "None detected",
        "Missing Skills": ", ".join(missing) or "None detected"
    }

print("Resume Screening Engine Ready!")

Resume Screening Engine Ready!


In [3]:
job = """
Python developer with SQL, Flask, Git and Pandas skills.
"""

resume = """
I know Python, SQL, Flask and Pandas.
I have used these skills in college projects.
"""

result = screen_resume(resume, job)
print(result)

{'Match Score (%)': 80, 'Matched Skills': 'flask, pandas, python, sql', 'Missing Skills': 'git'}


In [4]:
import fitz
from google.colab import files

uploaded = files.upload()

resumes = {}

for filename, file_bytes in uploaded.items():
    if filename.lower().endswith(".pdf"):
        pdf = fitz.open(stream=file_bytes, filetype="pdf")
        text = "\n".join(page.get_text() for page in pdf)
        resumes[filename] = text
        pdf.close()
        print(f"Successfully read: {filename}")

print("Total PDF resumes:", len(resumes))

C:\Users\Dharma\Desktop\AI Resume Project\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
job_description = """
Python developer with SQL, Flask, Git and Pandas skills.
"""

results = []

for filename, resume_text in resumes.items():
    result = screen_resume(resume_text, job_description)

    results.append({
        "Resume": filename,
        **result
    })

if results:
    df = pd.DataFrame(results)
    df = df.sort_values(
        by="Match Score (%)",
        ascending=False
    )
    display(df)
else:
    print("No readable PDF resumes found. Upload PDFs first.")

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr
import pandas as pd

def show_dashboard(job_description):
    rows = []

    for filename, resume_text in resumes.items():
        if not filename.lower().endswith(".pdf"):
            continue

        result = screen_resume(resume_text, job_description)

        rows.append({
            "Candidate": filename.replace(".pdf", ""),
            "Match Score (%)": result["Match Score (%)"],
            "Matched Skills": result["Matched Skills"],
            "Missing Skills": result["Missing Skills"]
        })

    if not rows:
        return pd.DataFrame(
            columns=["Candidate", "Match Score (%)",
                     "Matched Skills", "Missing Skills"]
        )

    return pd.DataFrame(rows).sort_values(
        "Match Score (%)", ascending=False
    )

demo = gr.Interface(
    fn=show_dashboard,
    inputs=gr.Textbox(
        label="Job Description",
        value="Python developer with SQL, Flask, Git and Pandas skills.",
        lines=4
    ),
    outputs=gr.Dataframe(label="Resume Screening Results"),
    title="AI Resume Screening System",
    description=(
        "Hackathon prototype: compare uploaded resumes "
        "against job requirements. Scores are decision-support "
        "indicators, not hiring decisions."
    )
)

demo.launch(share=True)

In [ ]:
def show_dashboard(job_description):
    rows = []

    for filename, resume_text in resumes.items():
        if not filename.lower().endswith(".pdf"):
            continue

        result = screen_resume(resume_text, job_description)

        rows.append({
            "Candidate": filename.replace(".pdf", ""),
            "Match Score (%)": result["Match Score (%)"],
            "Matched Skills": result["Matched Skills"],
            "Missing Skills": result["Missing Skills"]
        })

    if not rows:
        return pd.DataFrame(columns=[
            "Candidate", "Match Score (%)",
            "Matched Skills", "Missing Skills"
        ])

    return pd.DataFrame(rows).sort_values(
        "Match Score (%)", ascending=False
    ).reset_index(drop=True)

In [ ]:
import gradio as gr
import pandas as pd

def show_dashboard(job_description):
    rows = []

    for filename, resume_text in resumes.items():
        if not filename.lower().endswith(".pdf"):
            continue

        result = screen_resume(resume_text, job_description)

        rows.append({
            "Candidate": filename.replace(".pdf", ""),
            "Match Score (%)": result["Match Score (%)"],
            "Matched Skills": result["Matched Skills"],
            "Missing Skills": result["Missing Skills"]
        })

    if not rows:
        return pd.DataFrame(
            columns=["Candidate", "Match Score (%)",
                     "Matched Skills", "Missing Skills"]
        )

    return pd.DataFrame(rows).sort_values(
        "Match Score (%)", ascending=False
    )

demo = gr.Interface(
    fn=show_dashboard,
    inputs=gr.Textbox(
        label="Job Description",
        value="Python developer with SQL, Flask, Git and Pandas skills.",
        lines=4
    ),
    outputs=gr.Dataframe(label="Resume Screening Results"),
    title="AI Resume Screening System",
    description=(
        "Hackathon prototype: compare uploaded resumes "
        "against job requirements. Scores are decision-support "
        "indicators, not hiring decisions."
    )
)

demo.launch(share=True)

In [ ]:
!pip -q install gradio plotly

import gradio as gr
import pandas as pd
import plotly.express as px
import html

def analyze_candidates(job_description):
    rows = []

    if not job_description.strip():
        return pd.DataFrame(), None, "<p>Please enter a job description.</p>"

    for filename, resume_text in resumes.items():
        if not filename.lower().endswith(".pdf"):
            continue

        result = screen_resume(resume_text, job_description)

        rows.append({
            "Candidate": filename.replace(".pdf", ""),
            "Match Score (%)": result["Match Score (%)"],
            "Matched Skills": result["Matched Skills"],
            "Missing Skills": result["Missing Skills"]
        })

    if not rows:
        return pd.DataFrame(), None, "<p>No resumes found. Upload PDFs first.</p>"

    df = pd.DataFrame(rows)
    df = df.sort_values("Match Score (%)", ascending=False).reset_index(drop=True)

    total = len(df)
    average = round(df["Match Score (%)"].mean())
    top = html.escape(df.iloc[0]["Candidate"])
    top_score = int(df.iloc[0]["Match Score (%)"])

    summary = f"""
    <div style="display:flex;flex-wrap:wrap;gap:14px;margin:12px 0">
      <div style="flex:1;min-width:145px;background:#172554;color:white;padding:20px;border-radius:14px">
        <div>Total Candidates</div>
        <h2>{total}</h2>
      </div>
      <div style="flex:1;min-width:145px;background:#164e63;color:white;padding:20px;border-radius:14px">
        <div>Average Skill Match</div>
        <h2>{average}%</h2>
      </div>
      <div style="flex:1;min-width:180px;background:#14532d;color:white;padding:20px;border-radius:14px">
        <div>Highest Matching Resume</div>
        <h3>{top}</h3>
        <h2>{top_score}%</h2>
      </div>
    </div>
    <p style="color:gray">Scores reflect detected skills only; they are not hiring decisions.</p>
    """

    chart = px.bar(
        df,
        x="Candidate",
        y="Match Score (%)",
        title="Candidate Skill Match Comparison",
        text="Match Score (%)",
        range_y=[0, 110],
        color="Match Score (%)",
        color_continuous_scale="Blues"
    )
    chart.update_layout(
        xaxis_title="Resume",
        yaxis_title="Skill Match (%)",
        showlegend=False
    )
    chart.update_traces(texttemplate="%{text}%", textposition="outside")

    return df, chart, summary


with gr.Blocks(theme=gr.themes.Soft(), title="AI Resume Screening System") as demo:
    gr.Markdown("""
    # 🧠 AI Resume Screening System
    ### Smart Recruitment • Resume Comparison • Skills Analysis

    Upload resumes using the earlier PDF upload cell, then compare candidates
    against the job requirements below.
    """)

    job_input = gr.Textbox(
        label="Job Description",
        value="Python developer with SQL, Flask, Git and Pandas skills.",
        lines=4,
        placeholder="Enter required skills and job responsibilities..."
    )

    analyze_btn = gr.Button("🔍 Analyze Resumes", variant="primary")

    summary_output = gr.HTML(label="Overview")

    with gr.Tab("📊 Dashboard"):
        chart_output = gr.Plot(label="Candidate Comparison")

    with gr.Tab("📋 Candidate Results"):
        table_output = gr.Dataframe(
            label="Ranked Candidates",
            interactive=False,
            wrap=True
        )

    analyze_btn.click(
        fn=analyze_candidates,
        inputs=[job_input],
        outputs=[table_output, chart_output, summary_output]
    )

    gr.Markdown("""
    ---
    **Prototype notice:** Match scores use transparent, rule-based skill matching.
    Recruiters must review resumes independently. Scores should not be used
    as the sole basis for hiring decisions.
    """)

demo.launch(share=True)